# 🏡 Vancouver Real Estate & Climate Regression Analysis
### Portfolio Project A - Statistical Inference & Quantitative Modeling

This project demonstrates a production-grade data science workflow integrating two distinct datasets: Vancouver residential housing transactions and monthly climate data. 

## 🎯 Objective
1. **Clean and preprocess** data using robust statistical methods (outlier removal via the **1.5 * IQR** rule).
2. **Merge** transaction-level data with monthly climate indicators.
3. **Build a Multiple Linear Regression model** using `statsmodels` to evaluate the impact of structural characteristics (bedrooms), location (distance to beach), and seasonal climate (monthly precipitation) on real estate prices.
4. **Validate OLS assumptions** (multicollinearity, homoscedasticity, residual distribution).
5. **Verify the Normal Equation** $\hat{\beta} = (X^T X)^{-1} X^T Y$ using matrix operations in `numpy` (CS229 core logic).

## 🛠️ Step 1: Import Dependencies & Load Data

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load raw datasets
df_housing = pd.read_csv("raw_data/vancouver_housing.csv")
df_climate = pd.read_csv("raw_data/vancouver_climate.csv")

print(f"Housing Transactions: {len(df_housing)} rows")
print(f"Climate Records: {len(df_climate)} rows")

## 🧹 Step 2: Data Cleaning, Outlier Filtering & Merging
We extract `year_month` from the transaction date to join with climate data. Then, we apply the **1.5 * IQR** rule to clean extreme pricing outliers.

In [ ]:
# Add Year-Month column
df_housing["year_month"] = pd.to_datetime(df_housing["date"]).dt.strftime("%Y-%m")

# Merge housing and climate data
df_combined = pd.merge(df_housing, df_climate, on="year_month", how="left")

# Outlier Removal (1.5 * IQR)
Q1 = df_combined["price_cad"].quantile(0.25)
Q3 = df_combined["price_cad"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_cleaned = df_combined[(df_combined["price_cad"] >= lower_bound) & (df_combined["price_cad"] <= upper_bound)]
df_outliers = df_combined[(df_combined["price_cad"] < lower_bound) | (df_combined["price_cad"] > upper_bound)]

print(f"Original dataset size: {len(df_combined)}")
print(f"Cleaned dataset size: {len(df_cleaned)}")
print(f"Outliers removed: {len(df_outliers)}")
print(f"Lower bound: {lower_bound:,.2f} CAD | Upper bound: {upper_bound:,.2f} CAD")

## 📊 Step 3: Exploratory Data Analysis (EDA)
Let's compare the pricing distributions before and after outlier removal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_housing["price_cad"] / 1e6, bins=40, kde=True, ax=axes[0], color="#f43f5e")
axes[0].set_title("Distribution Before Outlier Removal", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Price (Millions CAD)")
axes[0].set_ylabel("Count")

sns.histplot(df_cleaned["price_cad"] / 1e6, bins=40, kde=True, ax=axes[1], color="#10b981")
axes[1].set_title("Distribution After 1.5*IQR Outlier Removal", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Price (Millions CAD)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 📈 Step 4: Fit Multiple OLS Regression Model
We regress house price on `distance_to_beach_km`, `precipitation_mm`, and `bedrooms` using `statsmodels`.

In [ ]:
Y = df_cleaned["price_cad"]
X = df_cleaned[["distance_to_beach_km", "precipitation_mm", "bedrooms"]]

# Add constant term (intercept beta_0)
X_with_const = sm.add_constant(X)

ols_model = sm.OLS(Y, X_with_const)
ols_results = ols_model.fit()

print(ols_results.summary())

### 🔍 Model Diagnostic Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Regression fit line vs Distance to beach
scatter = axes[0].scatter(
    df_cleaned["distance_to_beach_km"], 
    df_cleaned["price_cad"] / 1e6, 
    c=df_cleaned["bedrooms"], 
    cmap="viridis", 
    alpha=0.6
)
fig.colorbar(scatter, ax=axes[0], label="Bedrooms")

# Trend line calculations
b_beach = ols_results.params["distance_to_beach_km"]
intercept = ols_results.params["const"]
b_beds = ols_results.params["bedrooms"]
b_precip = ols_results.params["precipitation_mm"]

x_range = np.linspace(df_cleaned["distance_to_beach_km"].min(), df_cleaned["distance_to_beach_km"].max(), 100)
y_pred_line = (intercept + b_beach * x_range + b_beds * df_cleaned["bedrooms"].mean() + b_precip * df_cleaned["precipitation_mm"].mean()) / 1e6

axes[0].plot(x_range, y_pred_line, color="#e11d48", linewidth=2.5, label="Regression Fit")
axes[0].set_title("Price vs. Distance to Beach", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Distance to Beach (km)")
axes[0].set_ylabel("Price (Millions CAD)")
axes[0].legend()

# 2. Residual plot
sns.scatterplot(x=ols_results.fittedvalues / 1e6, y=ols_results.resid / 1e3, alpha=0.5, color="#5b21b6", ax=axes[1])
axes[1].axhline(y=0, color="#ef4444", linestyle="--", linewidth=2)
axes[1].set_title("Residuals vs. Fitted Values", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Fitted Values (Millions CAD)")
axes[1].set_ylabel("Residuals (Thousands CAD)")

plt.tight_layout()
plt.show()

## 🧮 Step 5: CS229 Normal Equation Verification
We solve OLS analytically using the Normal Equation:
$$\hat{\beta} = (X^T X)^{-1} X^T Y$$
Then, we compare the numpy output against the statsmodels coefficients.

In [ ]:
# Prepare vectors
y_vec = df_cleaned["price_cad"].values
ones = np.ones(len(df_cleaned))
X_features = df_cleaned[["distance_to_beach_km", "precipitation_mm", "bedrooms"]].values
X_matrix = np.column_stack((ones, X_features))

# Calculate Normal Equation step-by-step
XTX = X_matrix.T @ X_matrix
XTX_inv = np.linalg.inv(XTX)
XTY = X_matrix.T @ y_vec

beta_hat = XTX_inv @ XTY

# Compare with Statsmodels coefficients
sm_coefs = ols_results.params.values
feature_names = ["Intercept", "Beach Distance (km)", "Precipitation (mm)", "Bedrooms"]

print("="*75)
print(f"{'Feature':<25} | {'Numpy Normal Equation':<22} | {'Statsmodels OLS':<22}")
print("-"*75)
for name, numpy_c, sm_c in zip(feature_names, beta_hat, sm_coefs):
    print(f"{name:<25} | {numpy_c:<22,.5f} | {sm_c:<22,.5f}")
print("="*75)

# Run assertion
assert np.allclose(beta_hat, sm_coefs, rtol=1e-5), "Error: NumPy Normal Equation does not match Statsmodels!"
print("✅ SUCCESS: Numerical verification complete! The OLS estimates match perfectly.")